# 16 — Batch inference cho 4 demo studies mới

Notebook xử lý 4 study mới đã export, tạo đủ prediction Phase 3 để dashboard hiển thị 12 xác suất.

Theo cấu trúc project hiện tại trong VS Code, notebook tự ưu tiên quét:
`<project-root>/demo/input/demo_sample/study_001..study_005/`.
Bạn không cần chuyển dữ liệu sang `C:\rsna_demo\studies`.

Notebook sẽ:
1. Tự phát hiện 5 package theo `manifest.json`.
2. Loại sample đầu tiên bằng StudyInstanceUID đã cấu hình.
3. Kiểm tra đủ `radiology_report.txt` cho 4 package còn lại.
4. Chạy preprocessing DINOv2 + ViT, DINOv2 5-fold inference, ViT inference và Phase 3 blend.
5. Lưu từng bundle dưới `demo/output/ensemble/<StudyInstanceUID>/`.

**Không train lại model.** Checkpoint được strict-load từ artifact hiện có trong `artifacts/`.


In [4]:
from pathlib import Path
import sys
import json
import importlib.util
import pandas as pd

# Tự tìm project root khi notebook được mở từ thư mục notebooks/.
def locate_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "rsna_knee" / "dicom" / "sample.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy project root chứa src/rsna_knee/dicom/sample.py. "
        "Hãy mở notebook bên trong project RSNA local."
    )

PROJECT_ROOT = locate_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Helper script được đóng gói riêng trong thư mục scripts/. Hỗ trợ cả cấu trúc
# ZIP mới (phẳng) và ZIP cũ có thư mục rsna_multi_study_inference_patch lồng bên trong.
SCRIPT_RELATIVE = Path("scripts") / "run_demo_batch_inference.py"
script_candidates = [
    PROJECT_ROOT / SCRIPT_RELATIVE,
    PROJECT_ROOT / "rsna_multi_study_inference_patch" / SCRIPT_RELATIVE,
    PROJECT_ROOT / "notebooks" / "run_demo_batch_inference.py",
    Path.cwd().resolve() / SCRIPT_RELATIVE,
]
SCRIPT_PATH = next((p.resolve() for p in script_candidates if p.is_file()), None)

if SCRIPT_PATH is None:
    checked = "\n".join(f"- {p}" for p in script_candidates)
    raise FileNotFoundError(
        "Không tìm thấy helper script run_demo_batch_inference.py.\n"
        "Bạn cần giải nén toàn bộ package batch inference vào project root, "
        "không chỉ copy riêng notebook. File phải nằm tại:\n"
        f"{PROJECT_ROOT / SCRIPT_RELATIVE}\n\n"
        "Các vị trí đã kiểm tra:\n" + checked
    )

# Import trực tiếp từ đường dẫn tuyệt đối để không phụ thuộc working directory/sys.path.
module_name = "_rsna_demo_batch_inference_runner"
spec = importlib.util.spec_from_file_location(module_name, SCRIPT_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Không thể tạo module spec từ: {SCRIPT_PATH}")
runner_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = runner_module
spec.loader.exec_module(runner_module)

FIRST_SAMPLE_UID = runner_module.FIRST_SAMPLE_UID
discover_study_packages = runner_module.discover_study_packages
_study_uid_from_package = runner_module._study_uid_from_package
run_batch_inference = runner_module.run_batch_inference

print("Project root:", PROJECT_ROOT)
print("Batch runner:", SCRIPT_PATH)
print("First sample UID excluded:", FIRST_SAMPLE_UID)


Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Batch runner: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\scripts\run_demo_batch_inference.py
First sample UID excluded: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149


In [5]:
# =========================
# CONFIGURATION — EDIT HERE ONLY IF NEEDED
# =========================

# Your screenshot shows this exact layout:
# <project-root>/demo/input/demo_sample/study_001..study_005/<manifest.json, dicom/, ...>
# Prefer that directory automatically. Fall back to the older short path only
# when the project-local directory does not contain exported study packages.

def _contains_exported_study(root: Path) -> bool:
    root = Path(root).expanduser()
    if not root.is_dir():
        return False
    for manifest_path in root.rglob("manifest.json"):
        package_root = manifest_path.parent
        if (
            (package_root / "series_metadata.csv").is_file()
            and (package_root / "series_selection.csv").is_file()
            and (package_root / "dicom").is_dir()
        ):
            return True
    return False

STUDIES_ROOT_CANDIDATES = [
    PROJECT_ROOT / "demo" / "input" / "demo_sample",  # matches your screenshot
    Path(r"C:\rsna_demo\studies"),                  # older recommended layout
    Path(r"C:\rsna_demo\demo_sample"),             # original single-sample location
]

STUDIES_ROOT = next(
    (candidate.resolve() for candidate in STUDIES_ROOT_CANDIDATES
     if _contains_exported_study(candidate)),
    STUDIES_ROOT_CANDIDATES[0].resolve(),
)

# Exactly four new studies; sample 001 is excluded by its StudyInstanceUID.
MAX_STUDIES = 4

# Leave blank for automatic artifact discovery under PROJECT_ROOT/artifacts.
DINO_BUNDLE_SOURCE = ""
VIT_ARTIFACT_SOURCE = ""

# None = CUDA if available, otherwise CPU.
DEVICE = None

# Safe CPU defaults.
DINO_ENCODER_CHUNK_SIZE = 1
VIT_SLICE_CHUNK_SIZE = 2

# Skip a study that already has a complete, UID-matching Phase 3 bundle.
SKIP_COMPLETE_STUDIES = True

# Exclude the first sample regardless of folder name.
EXCLUDE_UIDS = (FIRST_SAMPLE_UID,)

print("Studies root:", STUDIES_ROOT)
print("Exists:", STUDIES_ROOT.is_dir())
print("Contains exported packages:", _contains_exported_study(STUDIES_ROOT))


Studies root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\input\demo_sample
Exists: True
Contains exported packages: True


In [6]:
# Preview before loading any model: should discover 5 packages, then exclude study_001.
package_roots = discover_study_packages(STUDIES_ROOT)
preview_rows = []
for package_root in package_roots:
    uid = _study_uid_from_package(package_root)
    if uid in EXCLUDE_UIDS:
        continue
    manifest = json.loads((package_root / "manifest.json").read_text(encoding="utf-8"))
    report_path = package_root / "radiology_report.txt"
    preview_rows.append({
        "folder": package_root.name,
        "StudyInstanceUID": uid,
        "split": manifest.get("source", {}).get("split", "not_recorded"),
        "radiology_report": report_path.is_file(),
        "package_path": str(package_root),
    })

if not preview_rows:
    raise RuntimeError(
        f"Không tìm thấy study package nào trong {STUDIES_ROOT}. "
        "Mỗi study_00x phải chứa trực tiếp manifest.json, series_metadata.csv, "
        "series_selection.csv và thư mục dicom/."
    )

preview_df = pd.DataFrame(preview_rows).sort_values("folder").reset_index(drop=True)
display(preview_df)

print(f"\nDiscovered packages (including excluded sample): {len(package_roots)}")
print(f"New studies after excluding first sample: {len(preview_df)}")

if len(preview_df) != MAX_STUDIES:
    raise RuntimeError(
        f"Expected {MAX_STUDIES} new study packages after excluding FIRST_SAMPLE_UID, "
        f"found {len(preview_df)}. Check that STUDIES_ROOT points to the folder "
        "containing study_001, study_002, study_003, study_004 and study_005."
    )
if preview_df["StudyInstanceUID"].duplicated().any():
    raise RuntimeError("Có StudyInstanceUID bị trùng trong thư mục studies.")
if not preview_df["radiology_report"].all():
    missing_reports = preview_df.loc[~preview_df["radiology_report"], "folder"].tolist()
    raise RuntimeError(
        "Thiếu radiology_report.txt trong các package: " + ", ".join(missing_reports) +
        ". Hãy bổ sung report trước khi chạy batch inference."
    )


,folder,StudyInstanceUID,split,radiology_report,package_path
0,study_002,1.2.826.0.1.3680043.8.498.11548045715264151632...,gold_validation,True,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...
1,study_003,1.2.826.0.1.3680043.8.498.72853333220043794904...,gold_validation,True,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...
2,study_004,1.2.826.0.1.3680043.8.498.56512564464301544960...,gold_validation,True,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...
3,study_005,1.2.826.0.1.3680043.8.498.11287937729196958426...,gold_validation,True,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...



Discovered packages (including excluded sample): 5
New studies after excluding first sample: 4


## Chạy batch inference

Cell dưới đây chạy đủ Phase 2 + Phase 3 cho bốn study mới. Không cần chạy thủ công từng notebook cho từng study.

Đầu ra được ghi tại `demo/output/ensemble/<StudyInstanceUID>/`, đúng vị trí dashboard multi-study đang tìm.


In [7]:
batch_rows = run_batch_inference(
    studies_root=STUDIES_ROOT,
    project_root=PROJECT_ROOT,
    dino_bundle_source=DINO_BUNDLE_SOURCE or None,
    vit_artifact_source=VIT_ARTIFACT_SOURCE or None,
    max_studies=MAX_STUDIES,
    exclude_uids=EXCLUDE_UIDS,
    device=DEVICE,
    dino_encoder_chunk_size=DINO_ENCODER_CHUNK_SIZE,
    vit_slice_chunk_size=VIT_SLICE_CHUNK_SIZE,
    skip_complete=SKIP_COMPLETE_STUDIES,
)

batch_df = pd.DataFrame(batch_rows)
display(batch_df.drop(columns=["traceback"], errors="ignore"))

print("\nPhase 3 output root:", PROJECT_ROOT / "demo" / "output" / "ensemble")

Project: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Studies root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\input\demo_sample
Device: cpu
DINOv2 bundle: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle
ViT artifact: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\checkpoints\102526\vit_pretrained_centerprior_model_artifacts.zip
Eligible new studies: 4
Skipped package paths: 1

[1/4] Loading 1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935 from C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\input\demo_sample\study_002


C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Fold 1/5: strict_load=True | encoded_windows=50/50 | range=[0.3442, 0.6195] | elapsed=14.3s
Fold 2/5: strict_load=True | encoded_windows=50/50 | range=[0.3011, 0.8640] | elapsed=13.5s
Fold 3/5: strict_load=True | encoded_windows=50/50 | range=[0.4012, 0.9036] | elapsed=13.3s
Fold 4/5: strict_load=True | encoded_windows=50/50 | range=[0.2872, 0.8761] | elapsed=14.4s
Fold 5/5: strict_load=True | encoded_windows=50/50 | range=[0.1282, 0.9390] | elapsed=13.6s
Five-fold inference complete | study=1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935 | valid_windows=50/50 | available_roles=5/5 | elapsed=70.6s
[1/4] PASS 1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935 | blend=weighted_ensemble | output=C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\ensemble\1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935

[2/4] Loading 1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921 from C:\Users\THINH DINH\Desktop\sp

,study_index,study_uid,package_root,status,message,has_radiology_report,dino_valid_windows,dino_available_roles,vit_valid_slices,dino_status,vit_status,ensemble_status
0,1,1.2.826.0.1.3680043.8.498.11548045715264151632...,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...,success,Saved 4 Phase 3 files.,True,50,5,36,ok,success,weighted_ensemble
1,2,1.2.826.0.1.3680043.8.498.72853333220043794904...,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...,success,Saved 4 Phase 3 files.,True,50,5,36,ok,success,weighted_ensemble
2,3,1.2.826.0.1.3680043.8.498.56512564464301544960...,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...,success,Saved 4 Phase 3 files.,True,50,5,36,ok,success,weighted_ensemble
3,4,1.2.826.0.1.3680043.8.498.11287937729196958426...,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...,success,Saved 4 Phase 3 files.,True,40,4,36,ok,success,weighted_ensemble



Phase 3 output root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\ensemble


In [8]:
# Final verification: independently check all four Phase 3 files and Study UID.
ENSEMBLE_ROOT = PROJECT_ROOT / "demo" / "output" / "ensemble"
required = [
    "ensemble_prediction.csv",
    "ensemble_prediction_long.csv",
    "ensemble_prediction.npz",
    "ensemble_report.json",
]

verification = []
for _, item in preview_df.iterrows():
    uid = str(item["StudyInstanceUID"])
    folder = ENSEMBLE_ROOT / uid
    missing = [name for name in required if not (folder / name).is_file()]
    report = {}
    if not missing:
        report = json.loads((folder / "ensemble_report.json").read_text(encoding="utf-8"))
    verification.append({
        "StudyInstanceUID": uid,
        "files_complete": not missing,
        "report_uid_matches": str(report.get("study_instance_uid", "")) == uid if not missing else False,
        "blend_status": report.get("blend_status", report.get("status", "")) if not missing else "",
        "missing_files": ", ".join(missing),
        "prediction_folder": str(folder),
    })

verification_df = pd.DataFrame(verification)
display(verification_df)

if not verification_df["files_complete"].all() or not verification_df["report_uid_matches"].all():
    raise RuntimeError(
        "Một số study chưa có Phase 3 bundle hợp lệ. Xem multi_study_inference_log.csv "
        "để biết study nào thất bại và lỗi cụ thể."
    )

print("BATCH PHASE 2 + PHASE 3: ALL FOUR STUDIES HAVE VALID OUTPUT BUNDLES.")

,StudyInstanceUID,files_complete,report_uid_matches,blend_status,missing_files,prediction_folder
0,1.2.826.0.1.3680043.8.498.11548045715264151632...,True,True,weighted_ensemble,,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...
1,1.2.826.0.1.3680043.8.498.72853333220043794904...,True,True,weighted_ensemble,,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...
2,1.2.826.0.1.3680043.8.498.56512564464301544960...,True,True,weighted_ensemble,,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...
3,1.2.826.0.1.3680043.8.498.11287937729196958426...,True,True,weighted_ensemble,,C:\Users\THINH DINH\Desktop\space\rsna-knee-ab...


BATCH PHASE 2 + PHASE 3: ALL FOUR STUDIES HAVE VALID OUTPUT BUNDLES.


## Dashboard

Sau khi notebook báo:

```text
BATCH PHASE 2 + PHASE 3: ALL FOUR STUDIES HAVE VALID OUTPUT BUNDLES.
```

khởi động lại dashboard:

```powershell
streamlit run app.py
```

Để `Prediction output root` ở:

```text
<project-root>\demo\output\ensemble
```

Dashboard sẽ ghép prediction theo `StudyInstanceUID` và hiển thị 12 xác suất cho từng study. Nếu một study đã có output cũ hợp lệ, notebook sẽ skip study đó khi `SKIP_COMPLETE_STUDIES=True`.

### Lưu ý diễn giải

Đây là inference demo trên các study thuộc `gold_validation`, không phải đánh giá độc lập. Các xác suất là đầu ra mô hình, không phải chẩn đoán hoặc nguy cơ lâm sàng đã được hiệu chuẩn.